# Launch & finish DPS runs of HLS multi-spectral composites (v2)

Builds per-tile HLS **multi-spectral composites** (mscomps) for a list of years, finds which tiles are still **missing**, **failed**, or flagged for a **redo**, resubmits only those, and rebuilds the tile index (tindex) that `run_boreal_biomass_map` and `run_extract_atl08_covars` read through `mosaiclib.HLS_TINDEX_FN_DICT`.

**Run top to bottom. The only cell you edit is `1. Run configuration`.** Nothing is submitted unless `DRY_RUN = False`.

| Step | What it does | Touches DPS? |
|---|---|---|
| 2. Status | Reads each year's HLS tindex: tiles **have** vs **missing** | no |
| 3. Fails | Latest job per tile from saved submission logs → which tiles **failed** (and are still not in the tindex) | reads job status |
| 4. Plan | Per year: tiles to submit (`DO_MISSING` / `DO_FAILS` / `DO_REDO`). Saved as CSV. | no |
| 5. Submit | Submits the plan in batches with pauses (CMR rate limit), only if `DRY_RUN = False` | **yes** |
| 6. Assess | Status counts for the submission just made, and the error message of failed jobs | reads job status |
| 7. Rebuild tindex | After jobs finish: backs up, rebuilds (newest comp per tile wins), verifies the count did not shrink | no |
| 8. Check composites | For a list of tiles: is there a comp in each year's tindex, and does it look sound (size, CRS, coverage, valid pixels, n obs) | no |
| Appendix | Register the algorithm, local test, parameter-table maps, delete duplicate comp dirs | — |

**Typical cycle to finish a composite year**: `DO_MISSING=True` → plan → submit → wait → 6 → 7 → back to 2. Review the comps (section 8, `Review_maps.ipynb`), add bad tiles to `REDO_TILES`, then `DO_REDO=True` with adjusted parameters.

**Algorithms** (`ALGO`)
* `do_HLS_stack_3-1-2`, version `HLS_stack_2023_v1`: the production composites (`HLS_H30_2016` … `HLS_H30_2025` in `mosaiclib`).
* `run_build_ms_composite`, version `build_ms_composite_v2`: adds `STAT` (`max` / `pct`), `STAT_PCT` and `TARGET_SPECTRAL_INDEX` (`ndvi`, `evi2`). Its output stays separate under its own version (e.g. `mosaiclib` `2019max` = `HLS/build_ms_composite_v1/HLS_H30_2019`).

**How the tindex picks a comp.** For HLS, `build_tindex` searches every run under `dps_output/<alg>/<version>/<MS_COMP_NAME>/` (all `RUN_NAME`s, all dates) and keeps the **newest** file per tile. So a redo with different parameters (a different `RUN_NAME`) replaces the old comp in the tindex. It also means a bad newer comp replaces a good older one: delete the bad run's dirs (Appendix) before rebuilding.

**`DO_MISSING` vs `DO_FAILS` vs `DO_REDO`**
* **MISSING** = in `TILE_SET` but not in the tindex. Covers never-submitted *and* failed tiles, but only as current as the last tindex rebuild (step 7).
* **FAILS** = the tile's most recent job for this composite has status `Failed` (or never got a job id) *and* the tile is not in the tindex. Reads only the submission logs written by this notebook. Use it right after a run, before rebuilding the tindex.
* **REDO** = tiles in `REDO_TILES[year]`, found by visual review or AGBD trend checks, resubmitted **even though they are in the tindex**, usually with different parameters.
* Any combination: union. None: every tile in `TILE_SET` (a full run, which needs `CONFIRM_FULL_RUN = True`).

**Compositing notes** (from the original notebooks)
* `MAX_CLOUDS = 0` with `MIN_N_FILT_RESULTS = 50` is the Nov 2024 setting (`mn` in `RUN_NAME`): the minimum number of filtered HLS scenes wanted before the max-cloud limit is hit. A high cloud limit (≥ 75%) at high latitude can mean 175+ scenes over 3 seasons, long runs and void artifacts.
* Submitting thousands of comp jobs at once gets `CMR Search rate exceeded` / `Request Limit Exceeded` or `502 Bad Gateway` errors in the jobs. Submit in batches (`BATCH_SIZE`, `WAIT_MINS`) and resubmit the fails.
* `ModuleNotFoundError: No module named 'maap'` in a job after someone else's commits: re-register the algorithm (Appendix).
* HLS sensors: Landsat-8 from 2013-02-11, Sentinel-2 from 2015-06-23, Landsat-9 from 2021-09-27. Composites before 2016 are mostly L30.

Paul Montesano, PhD — reorganized Oct 2026 from `DPS_mscomp_stack.ipynb` and `DPS_build_ms_composite.ipynb`, same layout as `DPS_run_boreal_v2.ipynb`

In [ ]:
import os, sys, glob, json, shutil, time, datetime, subprocess
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
import matplotlib.pyplot as plt
from shapely.geometry import box

from maap.maap import MAAP
maap = MAAP(maap_host='api.maap-project.org')

# One clone for imports AND for the build_tindex script
ICESAT2_BOREAL_REPO_PATH = '/projects/my-private-bucket/code/icesat2_boreal'
sys.path.insert(0, ICESAT2_BOREAL_REPO_PATH + '/lib')
import ExtractUtils
import mosaiclib
from mosaiclib import HLS_TINDEX_FN_DICT, MINI_DATELINE_TILES, LARGE_DATELINE_TILES
print('ExtractUtils from', ExtractUtils.__file__)

## 1. Run configuration — the only cell to edit
Fill-in runs (`DO_MISSING`, `DO_FAILS`) should use the same `ALGO`, season and cloud parameters as the original run of that composite, so all tiles of a year are made the same way. Redo runs usually change them on purpose (e.g. a wider season).

In [ ]:
# --- What product -----------------------------------------------------------
ALGO = 'do_HLS_stack_3-1-2'          # 'do_HLS_stack_3-1-2' (production) or 'run_build_ms_composite'
ALGO_VERSIONS = {'do_HLS_stack_3-1-2': 'HLS_stack_2023_v1', 'run_build_ms_composite': 'build_ms_composite_v2'}
ALG_VERSION = ALGO_VERSIONS[ALGO]     # registered algorithm version (= git tag)
USER         = 'montesano'
WORKER_TYPE  = 'maap-dps-gedi_boreal_worker-32gb'
YEAR_LIST    = ['2025']               # composite years
YEAR_HALF_WINDOW = 0                  # 0: single-year comp (HLS_H30_2025). 1: 3-year comp 2024-2026, named HLS_H30_c2025

# --- Where --------------------------------------------------------------------
TILE_GRID  = 'boreal'                 # 'boreal', 'conus' or 'conus_centamerica' (see TILE_GRIDS below)
REGION_STR = ''                       # '' for boreal; e.g. 'CONUS' keeps a region's output separate (HLSCONUS_H30_2024)

# --- Compositing parameters ---------------------------------------------------
HLS_PRODUCT   = 'H30'                 # 'H30' (L30 + S30), 'L30' or 'S30'
SEASON_START  = '06-15'               # full runs used 07-01 / 08-31; the 2020-2025 redos used 06-15 / 09-15
SEASON_STOP   = '09-15'
MAX_CLOUDS    = 0
MIN_N_FILT_RESULTS = 50
COMP_PARAMS = {'sat_api': 'https://cmr.earthdata.nasa.gov/stac/LPCLOUD', 'tile_buffer_m': 0, 'composite_type': 'HLS', 'shape': 3000}
# run_build_ms_composite only
STAT = 'max'                          # 'max' or 'pct'
STAT_PCT = '50'                       # percentile when STAT = 'pct'
TARGET_SPECTRAL_INDEX = 'evi2'        # 'ndvi' or 'evi2'
THRESH_MIN_NDVI = -1

# --- Which tiles -------------------------------------------------------------
DO_MISSING = True        # tiles in TILE_SET not yet in the tindex
DO_FAILS   = False       # tiles whose latest job failed and are still not in the tindex
DO_REDO    = False       # tiles in REDO_TILES[year], even if they are in the tindex
CONFIRM_FULL_RUN = False # must be True to submit every tile in TILE_SET (all three flags False)
TILE_SET = None          # None = every tile in TILE_GRID; or a list of tile_num, e.g. mosaiclib.set_all_test_tiles
REFRESH_FAILS = True     # re-query job status; False = reuse the cached fails JSON

# Tiles to redo, found by review (AGBD trends 2020-2025, Review_maps.ipynb). Keep adding; note why.
REDO_TILES = {
    '2020': [1672, 1613, 1614, 1615, 1558, 1557, 3946, 2002, 2003, 1929, 1930, 1931, 1049, 34675, 1729, 1730]  # North America
            + [27, 38, 39] + [557],                                                                        # Scandinavia, Eurasia
    '2021': [1790, 1791, 1792, 1860, 1861, 943, 895, 34443, 34210, 988, 989, 990, 938, 939],               # North America
    '2023': [289],                                                                                          # Eurasia
    '2024': [3741, 3767, 3862, 2452, 2534] + [640],                                                         # North America, Eurasia
    '2025': [1726, 1727, 1794, 1795] + [126, 192],                                                          # North America, Eurasia
}

# --- Safety -----------------------------------------------------------------
DRY_RUN  = True          # True: build and save the plan only. False: submit in step 5.
MAX_JOBS = 6000          # refuse to submit more than this in one go
BATCH_SIZE = 5           # pause WAIT_MINS after every BATCH_SIZE jobs (CMR rate limit); None = no pauses
WAIT_MINS  = 5           # 5 per 5 min suits redo lists; for thousands of tiles that is days of submitting

# --- Paths --------------------------------------------------------------------
DPS_SUBMISSION_RESULTS_DIR = '/projects/my-public-bucket/dps_submission_results'
TINDEX_ROOT     = '/projects/my-public-bucket/DPS_tile_lists/HLS'          # tindex: {TINDEX_ROOT}/{ALG_VERSION}/{MS_COMP_NAME}
DPS_OUTPUT_ROOT = f'/projects/my-private-bucket/dps_output/{ALGO}/{ALG_VERSION}'
PLAN_DIR        = f'{DPS_SUBMISSION_RESULTS_DIR}/plans'
for d in (DPS_SUBMISSION_RESULTS_DIR, PLAN_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
# Derived names (do not edit)
TILE_GRIDS = {
    'boreal':            'boreal_tiles_v004_model_ready',
    'conus':             'tile_grid_conus',
    'conus_centamerica': 'tile_grid_conus_centamerica',
}
GRID = {'layer': TILE_GRIDS[TILE_GRID],
        'url':   f'https://maap-ops-workspace.s3.amazonaws.com/shared/montesano/databank/{TILE_GRIDS[TILE_GRID]}.gpkg',
        'local': f'/projects/shared-buckets/montesano/databank/{TILE_GRIDS[TILE_GRID]}.gpkg'}

def comp_years(year):
    w = YEAR_HALF_WINDOW
    return str(int(year) - w), str(int(year) + w)

def run_name(year):
    ys, ye = comp_years(year)
    if ALGO == 'run_build_ms_composite':
        stat_str = f'{STAT}{STAT_PCT}' if STAT == 'pct' else STAT
        return f'mc{MAX_CLOUDS:03}_mn{MIN_N_FILT_RESULTS:03}_{TARGET_SPECTRAL_INDEX}{stat_str}_{SEASON_START}_{SEASON_STOP}_{ys}_{ye}'
    return f'mc{MAX_CLOUDS}_mn{MIN_N_FILT_RESULTS}_{SEASON_START}_{SEASON_STOP}_{ys}_{ye}'

MS_COMP_NAMES = {y: f"HLS{REGION_STR}_{HLS_PRODUCT}_{'c' if YEAR_HALF_WINDOW else ''}{y}" for y in YEAR_LIST}
RUN_NAMES     = {y: run_name(y) for y in YEAR_LIST}
IDENTIFIERS   = {y: f'{MS_COMP_NAMES[y]}/{RUN_NAMES[y]}' for y in YEAR_LIST}
MODE = '+'.join([m for m, on in (('MISSING', DO_MISSING), ('FAILS', DO_FAILS), ('REDO', DO_REDO)) if on]) or 'ALL'
print(f'{ALGO} {ALG_VERSION} | grid {TILE_GRID} | years {YEAR_LIST[0]}–{YEAR_LIST[-1]} | mode {MODE} | DRY_RUN={DRY_RUN}')
for y, i in IDENTIFIERS.items():
    print('  ', i)

### Helper functions (no edits needed)

In [ ]:
def tindex_paths(year):
    d = f'{TINDEX_ROOT}/{ALG_VERSION}/{MS_COMP_NAMES[year]}'
    p = {'dir': d, 'csv': f'{d}/HLS_tindex_master.csv', 'gpkg': f'{d}/HLS_tindex_master.gpkg',
         'dups': f'{d}/HLS_tindex_master_duplicates.csv', 'mosaic': f'{d}/HLS_tindex_master_mosaic.json',
         'params': f'{d}/HLS_input_params.csv'}
    p['s3_csv'] = p['csv'].replace('/projects/my-public-bucket', f's3://maap-ops-workspace/shared/{USER}')
    return p

def read_tindex_tiles(year):
    """Tiles present in this year's tindex (empty set if no tindex yet)."""
    fn = tindex_paths(year)['csv']
    return set(pd.read_csv(fn).tile_num.astype(int)) if os.path.exists(fn) else set()

def get_in_param_dict(tile_num, year):
    ys, ye = comp_years(year)
    p = {'in_tile_fn': GRID['url'], 'in_tile_num': int(tile_num), 'in_tile_layer': GRID['layer'], **COMP_PARAMS,
         'start_year': ys, 'end_year': ye, 'start_month_day': SEASON_START, 'end_month_day': SEASON_STOP,
         'max_cloud': MAX_CLOUDS, 'hls_product': HLS_PRODUCT, 'min_n_filt_results': MIN_N_FILT_RESULTS}
    if ALGO == 'run_build_ms_composite':
        p.update({'thresh_min_ndvi': THRESH_MIN_NDVI, 'stat': STAT, 'stat_pct': STAT_PCT,
                  'target_spectral_index': TARGET_SPECTRAL_INDEX})
    return p

In [ ]:
def submission_logs(year):
    """Submission CSVs this notebook saved for this composite (any RUN_NAME), oldest first."""
    fns = glob.glob(f'{DPS_SUBMISSION_RESULTS_DIR}/DPS_{ALGO}_{MS_COMP_NAMES[year]}_*_submission_results_*.csv')
    return sorted(fns, key=lambda f: int(os.path.splitext(f)[0].split('_')[-1]))

def latest_jobs(year):
    """One row per tile: the most recent submission for this composite."""
    parts = []
    for order, fn in enumerate(submission_logs(year)):
        d = pd.read_csv(fn)
        d['log_file'], d['log_order'] = os.path.basename(fn), order
        parts.append(d)
    if not parts:
        return pd.DataFrame(columns=['tile_num', 'job_id', 'submit_status', 'log_file'])
    d = pd.concat(parts, ignore_index=True)
    d['tile_num'] = d.tile_num.astype(int)
    return d.sort_values(['log_order', 'dps_num']).drop_duplicates('tile_num', keep='last').reset_index(drop=True)

def job_status_table(jobs):
    """Adds a 'status' column via MAAP. Tiles that never got a job id are marked 'SubmitFailed'."""
    ok = jobs.job_id.notna() & (jobs.job_id.astype(str) != '')
    out = jobs.copy()
    out['status'] = 'SubmitFailed'
    if ok.any():
        st = ExtractUtils.BUILD_TABLE_JOBSTATUS(jobs[ok][['tile_num', 'job_id']])
        out.loc[ok, 'status'] = out.loc[ok, 'job_id'].map(dict(zip(st.job_id, st.status))).fillna('Unknown')
    return out

def job_error(job_id, n_lines=6):
    """Last lines of a failed job's error output (after the last Traceback)."""
    try:
        txt = str(maap.getJobResult(job_id)[-1])
    except Exception as e:
        return f'(could not get job result: {e})'
    tail = [l for l in txt.split('\nTraceback')[-1].splitlines() if l.strip()]
    return '\n'.join(tail[-n_lines:])

def job_errors(jt, n=10, n_lines=6):
    """Error tails for up to n failed jobs, grouped so repeated errors show once with their tiles."""
    bad = jt[jt.status == 'Failed'].head(n)
    errs = pd.DataFrame({'tile_num': bad.tile_num.values, 'job_id': bad.job_id.values,
                         'error': [job_error(j, n_lines) for j in bad.job_id]})
    for err, g in errs.groupby('error'):
        print(f'--- {len(g)} job(s), tiles {sorted(g.tile_num)}\n{err}\n')
    return errs

In [ ]:
def build_plan(year, identifier, tile_set, grid_tiles, have, failed, redo):
    """Tiles to submit for one composite year, with the reason each tile is (or is not) included."""
    missing = sorted(set(tile_set) - have)
    want = set()
    if DO_MISSING:
        want |= set(missing)
    if DO_FAILS:
        want |= set(failed)
    if DO_REDO:
        want |= set(redo)
    if not (DO_MISSING or DO_FAILS or DO_REDO):
        want = set(tile_set)
    dateline = set(MINI_DATELINE_TILES + LARGE_DATELINE_TILES)
    rows = []
    for t in sorted(want):
        runnable = t in grid_tiles
        rows.append({'identifier': identifier, 'year': year, 'tile_num': t,
                     'is_missing': t in missing, 'is_failed': t in failed, 'is_redo': t in redo, 'in_tindex': t in have,
                     'is_dateline': t in dateline, 'runnable': runnable,
                     'reason_not_runnable': '' if runnable else f'not in tile grid {GRID["layer"]}'})
    return pd.DataFrame(rows, columns=['identifier', 'year', 'tile_num', 'is_missing', 'is_failed', 'is_redo', 'in_tindex',
                                       'is_dateline', 'runnable', 'reason_not_runnable'])

def submit_identifier(identifier, year, tiles):
    """Submits in batches; the log CSV is rewritten after every batch so a dead kernel doesn't lose it."""
    now = pd.Timestamp.now().strftime('%Y%m%d%H%M')
    fn = f"{DPS_SUBMISSION_RESULTS_DIR}/DPS_{ALGO}_{identifier.replace('/', '_')}_submission_results_{len(tiles)}_{now}.csv"
    rows = []
    for i, t in enumerate(tiles, 1):
        try:
            r = maap.submitJob(identifier=identifier, algo_id=ALGO, version=ALG_VERSION, username=USER, queue=WORKER_TYPE,
                               **get_in_param_dict(t, year))
            job_id, status = r.id, r.status
        except Exception as e:
            job_id, status = None, f'error: {e}'
        rows.append({'dps_num': i, 'tile_num': t, 'submit_time': datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S'),
                     'algo_id': ALGO, 'user': USER, 'worker_type': WORKER_TYPE, 'job_id': job_id,
                     'submit_status': status, 'IDENTIFIER': identifier, 'run_name': RUN_NAMES[year]})
        if i in (1, len(tiles)) or i % 100 == 0:
            print(f'  {i}/{len(tiles)} tile {t}: {status}')
        if BATCH_SIZE and i % BATCH_SIZE == 0 and i < len(tiles):
            pd.DataFrame(rows).to_csv(fn, index=False)
            print(f'  {i}/{len(tiles)} submitted ...waiting {WAIT_MINS} min')
            time.sleep(60 * WAIT_MINS)
    df = pd.DataFrame(rows)
    df.to_csv(fn, index=False)
    print(f'  saved {fn}')
    return df

In [ ]:
def rebuild_tindex(year, ENV='pangeo', DEBUG=False, WRITE_PARAMS_TABLE=False,
                   SCRIPT=f'{ICESAT2_BOREAL_REPO_PATH}/lib/build_tindex_master_refactored.py'):
    """Back up, rebuild with build_tindex_master_refactored.py, verify the tile count.

    HLS output paths have no run date in the search pattern: the script globs
    <user>/dps_output/<alg>/<ALG_VERSION>/<MS_COMP_NAME>/**/*.tif (every RUN_NAME and date) and keeps
    the newest file per tile. It OVERWRITES HLS_tindex_master.csv/.gpkg/_mosaic.json.
    WRITE_PARAMS_TABLE also writes HLS_input_params.csv (params + num scenes per tile; reads one json per comp, slow)."""
    tp = tindex_paths(year)
    n_before = len(read_tindex_tiles(year))
    stamp = pd.Timestamp.now().strftime('%Y%m%d%H%M%S')
    bdir = f"{tp['dir']}/backup_{stamp}"
    for fn in (tp['csv'], tp['gpkg'], tp['dups'], tp['mosaic'], tp['params']):
        if os.path.exists(fn):
            os.makedirs(bdir, exist_ok=True)
            shutil.copy2(fn, bdir)
    os.makedirs(tp['dir'], exist_ok=True)
    cmd = [f'/opt/conda/envs/{ENV}/bin/python', SCRIPT,
           '-t', 'HLS', '-alg_name', ALGO, '--user', USER,
           '--dps_identifier', f'{ALG_VERSION}/{MS_COMP_NAMES[year]}',
           '--outdir', tp['dir'], '-boreal_tile_index_path', GRID['local'],
           '--RETURN_DUPS', '--WRITE_TINDEX_MATCHES_GDF']
    if WRITE_PARAMS_TABLE:
        cmd.append('--WRITE_MSCOMP_DF')
    if DEBUG:
        cmd.append('--DEBUG')
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:   # the script exits 1 before writing when nothing is found, so the tindex is untouched
        print(res.stdout[-3000:], res.stderr[-3000:])
        raise RuntimeError(f'{os.path.basename(SCRIPT)} failed for {MS_COMP_NAMES[year]}; backup in {bdir}')
    if DEBUG:
        print(res.stdout[-3000:])
    n_after = len(read_tindex_tiles(year))
    flag = 'OK' if n_after >= n_before else '!! SHRANK — restore with restore_tindex(year, backup_dir)'
    print(f'  {MS_COMP_NAMES[year]}: tiles before {n_before} -> after {n_after}  {flag}   (backup: {bdir if n_before else "none"})')
    in_mosaiclib = HLS_TINDEX_FN_DICT.get(year) == tp['s3_csv']
    return {'year': year, 'ms_comp_name': MS_COMP_NAMES[year], 'n_before': n_before, 'n_after': n_after, 'backup_dir': bdir,
            'tindex': tp['s3_csv'], 'is_mosaiclib_HLS_TINDEX_FN_DICT[year]': in_mosaiclib}

def restore_tindex(year, backup_dir):
    for fn in glob.glob(f'{backup_dir}/*'):
        shutil.copy2(fn, tindex_paths(year)['dir'])
    print('restored', MS_COMP_NAMES[year], 'from', backup_dir)

In [ ]:
def comp_info(row, grid_gdf, n=300):
    """Size, CRS, tile coverage, valid fraction and median n obs of one composite (read from overviews, ~n x n pixels)."""
    fn = row['local_path'] if isinstance(row.get('local_path'), str) and os.path.exists(row['local_path']) else row['s3_path']
    out = {'file': os.path.basename(fn)}
    try:
        with rio.Env(AWS_NO_SIGN_REQUEST='YES'), rio.open(fn) as src:
            names = list(src.descriptions)
            out.update({'width': src.width, 'height': src.height, 'n_bands': src.count, 'crs': src.crs.to_string() if src.crs else None})
            tile = grid_gdf[grid_gdf.tile_num == row['tile_num']]
            if len(tile) and src.crs:
                g = tile.to_crs(src.crs).geometry.iloc[0]
                out['tile_coverage'] = round(g.intersection(box(*src.bounds)).area / g.area, 3)
            shape = (max(1, src.height * n // max(src.height, src.width)), max(1, src.width * n // max(src.height, src.width)))
            red = src.read(names.index('Red') + 1 if 'Red' in names else 3, out_shape=shape, masked=True).astype('float32')
            valid = ~np.ma.getmaskarray(red) & np.isfinite(red.filled(np.nan)) & (red.filled(-9999) != -9999)
            out['valid_frac'] = round(float(valid.mean()), 3)
            if 'count' in names:
                cnt = src.read(names.index('count') + 1, out_shape=shape, masked=True)
                out['n_obs_median'] = float(np.ma.median(np.ma.masked_where(~valid, cnt))) if valid.any() else 0.0
    except Exception as e:
        out['error'] = str(e)[:200]
    return out

def check_composites(tiles, years, grid_gdf, tindex_dict=HLS_TINDEX_FN_DICT):
    """One row per tile x year: is the tile in that year's tindex, and what its comp looks like."""
    rows = []
    for y in years:
        fn = tindex_dict.get(y)
        ti = pd.read_csv(fn) if fn else pd.DataFrame(columns=['tile_num'])
        ti['tile_num'] = ti.tile_num.astype(int)
        for t in tiles:
            r = ti[ti.tile_num == t]
            base = {'tile_num': t, 'year': y, 'in_tindex': len(r) > 0, 'n_rows': len(r)}
            if len(r):
                r = r.iloc[-1].to_dict()
                base.update({'creation_time': r.get('creation time'), 's3_path': r['s3_path'], **comp_info(r, grid_gdf)})
            rows.append(base)
    df = pd.DataFrame(rows).reindex(columns=['tile_num', 'year', 'in_tindex', 'n_rows', 'creation_time', 's3_path', 'file', 'width',
                                             'height', 'n_bands', 'crs', 'tile_coverage', 'valid_frac', 'n_obs_median', 'error'])
    flags = []
    for _, r in df.iterrows():
        f = []
        if not r.in_tindex: f.append('not in tindex')
        if r.n_rows > 1: f.append('duplicate tindex rows')
        if isinstance(r.error, str): f.append('cannot open')
        if pd.notna(r.width) and (r.width != COMP_PARAMS['shape'] or r.height != COMP_PARAMS['shape']): f.append('not 3000x3000')
        if pd.notna(r.tile_coverage) and r.tile_coverage < 0.95: f.append('does not cover tile')
        if pd.notna(r.valid_frac) and r.valid_frac < 0.5: f.append('mostly nodata')
        flags.append(', '.join(f))
    df['problems'] = flags
    return df

def show_comps(tile_num, years, tindex_dict=HLS_TINDEX_FN_DICT, bands=('SWIR', 'NIR', 'Red'), n=600):
    """False-color quick look of one tile's comp in each year (2-98% stretch, from overviews)."""
    fig, axes = plt.subplots(1, len(years), figsize=(5 * len(years), 5), squeeze=False)
    for ax, y in zip(axes[0], years):
        ax.set_axis_off()
        ti = pd.read_csv(tindex_dict[y]) if tindex_dict.get(y) else pd.DataFrame(columns=['tile_num'])
        r = ti[ti.tile_num.astype(int) == tile_num]
        if r.empty:
            ax.set_title(f'{y}: not in tindex'); continue
        r = r.iloc[-1]
        fn = r['local_path'] if isinstance(r.get('local_path'), str) and os.path.exists(r['local_path']) else r['s3_path']
        with rio.Env(AWS_NO_SIGN_REQUEST='YES'), rio.open(fn) as src:
            names = list(src.descriptions)
            idx = [names.index(b) + 1 if b in names else i for b, i in zip(bands, (5, 4, 3))]
            shape = (src.height * n // max(src.height, src.width), src.width * n // max(src.height, src.width))
            a = src.read(idx, out_shape=(3, *shape), masked=True).astype('float32').filled(np.nan)
            a[a == -9999] = np.nan
        lo, hi = np.nanpercentile(a, 2, axis=(1, 2)), np.nanpercentile(a, 98, axis=(1, 2))
        rgb = np.clip((a - lo[:, None, None]) / (hi - lo)[:, None, None], 0, 1).transpose(1, 2, 0)
        ax.imshow(np.nan_to_num(rgb)); ax.set_title(f'{tile_num} {y}\n{os.path.basename(fn)}', fontsize=8)
    plt.tight_layout()

## 2. Status: which tiles each year has and is missing
`in mosaiclib` = this tindex is the one `HLS_TINDEX_FN_DICT[year]` points to, i.e. the one `run_boreal_biomass_map` and `run_extract_atl08_covars` use. If it says `False`, add or update the entry in `lib/mosaiclib.py` once the composite is complete.

In [ ]:
grid = gpd.read_file(GRID['url'])
GRID_TILES = set(grid.tile_num.astype(int))
TILES = sorted(set(int(t) for t in (TILE_SET if TILE_SET is not None else GRID_TILES)))
HAVE = {y: read_tindex_tiles(y) for y in YEAR_LIST}
status = pd.DataFrame([{'year': y, 'ms_comp_name': MS_COMP_NAMES[y], 'n_target': len(TILES),
                        'n_have': len(set(TILES) & HAVE[y]), 'n_missing': len(set(TILES) - HAVE[y]),
                        'n_redo_listed': len(REDO_TILES.get(y, [])),
                        'in mosaiclib': HLS_TINDEX_FN_DICT.get(y) == tindex_paths(y)['s3_csv']} for y in YEAR_LIST])
status

In [ ]:
# Optional: map of missing tiles for one year
SHOW_MAP_YEAR = YEAR_LIST[0]
grid[grid.tile_num.isin(set(TILES) - HAVE[SHOW_MAP_YEAR])].explore(color='red', tooltip='tile_num')

## 3. Fails: latest job per tile from the saved submission logs
Only tiles whose **most recent** job for this composite failed (or never got a job id), and that are still not in the tindex. Reads the logs this notebook writes (`DPS_<ALGO>_<MS_COMP_NAME>_<RUN_NAME>_submission_results_*.csv`); for runs from the old notebooks use `DO_MISSING` after a tindex rebuild instead. Cached to JSON so `DO_FAILS` works when you come back without re-querying (`REFRESH_FAILS = False`).

In [ ]:
FAILS_JSON = f"{PLAN_DIR}/fails_{ALGO}_{ALG_VERSION}_HLS{REGION_STR}_{HLS_PRODUCT}.json"
JOB_TABLES = {}
if DO_FAILS and not REFRESH_FAILS and os.path.exists(FAILS_JSON):
    FAILED = {k: set(v) for k, v in json.load(open(FAILS_JSON)).items()}
    print('loaded cached fails from', FAILS_JSON)
elif DO_FAILS:
    FAILED = {}
    for y in YEAR_LIST:
        jobs = latest_jobs(y)
        if jobs.empty:
            FAILED[y] = set(); print(f'{MS_COMP_NAMES[y]}: no submission logs'); continue
        jt = job_status_table(jobs)
        JOB_TABLES[y] = jt
        bad = jt.status.isin(['Failed', 'SubmitFailed', 'Offline', 'Deleted'])
        FAILED[y] = set(jt.tile_num[bad].astype(int)) - HAVE[y]
        print(f'{MS_COMP_NAMES[y]}: latest jobs {len(jt)} | ' + ', '.join(f'{k} {v}' for k, v in jt.status.value_counts().items())
              + f' | failed & not in tindex: {len(FAILED[y])}')
    json.dump({y: sorted(v) for y, v in FAILED.items()}, open(FAILS_JSON, 'w'))
else:
    FAILED = {y: set() for y in YEAR_LIST}
    print('DO_FAILS = False: fails not computed')

In [ ]:
# Optional: why did they fail? Error tail of up to 10 failed jobs per year (CMR rate limits show up here)
for y, jt in JOB_TABLES.items():
    print(f'===== {MS_COMP_NAMES[y]}')
    job_errors(jt, n=10)

## 4. Plan: what will be submitted
Review this table before setting `DRY_RUN = False`. `is_dateline` marks tiles that touch the ±180° line (`mosaiclib.MINI_DATELINE_TILES` / `LARGE_DATELINE_TILES`); check those comps in section 8 after the run.

In [ ]:
if not (DO_MISSING or DO_FAILS or DO_REDO) and not CONFIRM_FULL_RUN:
    raise ValueError('DO_MISSING, DO_FAILS and DO_REDO are all False (full run of TILE_SET). Set CONFIRM_FULL_RUN = True if intended.')

plan = pd.concat([build_plan(y, IDENTIFIERS[y], TILES, GRID_TILES, HAVE[y], FAILED.get(y, set()), set(REDO_TILES.get(y, [])))
                  for y in YEAR_LIST], ignore_index=True)
plan = plan.astype({'tile_num': int, 'is_missing': bool, 'is_failed': bool, 'is_redo': bool, 'in_tindex': bool,
                    'is_dateline': bool, 'runnable': bool})
plan_summary = (plan.groupby(['year', 'identifier'])
                    .agg(n_selected=('tile_num', 'size'), n_missing=('is_missing', 'sum'), n_failed=('is_failed', 'sum'),
                         n_redo=('is_redo', 'sum'), n_dateline=('is_dateline', 'sum'),
                         n_not_runnable=('runnable', lambda s: int((~s).sum())), n_to_submit=('runnable', 'sum'))
                    .reset_index())
PLAN_FN = f"{PLAN_DIR}/plan_{ALGO}_HLS{REGION_STR}_{HLS_PRODUCT}_{MODE}_{pd.Timestamp.now():%Y%m%d%H%M}.csv"
plan.to_csv(PLAN_FN, index=False)
n_total = int(plan.runnable.sum())
batches = (n_total - 1) // BATCH_SIZE if BATCH_SIZE and n_total else 0
print('plan saved:', PLAN_FN, '| total to submit:', n_total, f'| ~{batches * WAIT_MINS / 60:.1f} h of batch pauses')
plan_summary

In [ ]:
# Not-runnable tiles and why
plan[~plan.runnable].groupby(['year', 'reason_not_runnable']).tile_num.apply(lambda s: sorted(s)).reset_index()

## 5. Submit (only if `DRY_RUN = False`)
With `BATCH_SIZE` set, this cell sleeps `WAIT_MINS` between batches and can run for a long time; the log CSV is saved after each batch.

In [ ]:
SUBMITTED = {}
if DRY_RUN:
    print(f'DRY_RUN: would submit {n_total} jobs. Set DRY_RUN = False in section 1 and re-run from the top.')
elif n_total > MAX_JOBS:
    raise ValueError(f'{n_total} jobs > MAX_JOBS={MAX_JOBS}. Raise MAX_JOBS if intended.')
else:
    for y, ident in IDENTIFIERS.items():
        tiles = plan[(plan.year == y) & plan.runnable].tile_num.tolist()
        print(f'\n{ident}: {len(tiles)} jobs')
        if tiles:
            SUBMITTED[y] = submit_identifier(ident, y, tiles)

## 6. Assess the submission just made
Re-run this cell until nothing is `Accepted`/`Running`; it then prints the error of failed jobs. To assess an older run in a new session, set `DO_FAILS = True` and run section 3, which reads the saved logs.

In [ ]:
if SUBMITTED:
    assess = pd.concat([job_status_table(df) for df in SUBMITTED.values()], ignore_index=True)
    display(assess.groupby(['IDENTIFIER', 'status']).size().unstack(fill_value=0))
    if not assess.status.isin(['Accepted', 'Running']).any() and (assess.status == 'Failed').any():
        job_errors(assess, n=10)
else:
    print('Nothing submitted in this session.')

## 7. Rebuild the tindex (after jobs finish)
Backs up the existing tindex, rebuilds it from every run of the composite (newest comp per tile wins), and reports before/after tile counts. Then go back to section 2.

In [ ]:
DO_REBUILD_TINDEX  = False
WRITE_PARAMS_TABLE = False   # also write HLS_input_params.csv (slow: one json read per comp); used by the Appendix maps
if DO_REBUILD_TINDEX:
    rebuilt = [rebuild_tindex(y, WRITE_PARAMS_TABLE=WRITE_PARAMS_TABLE) for y in YEAR_LIST]
    display(pd.DataFrame(rebuilt))

## 8. Check composites for a list of tiles
Answers "is there a comp for these tiles in these years, and is it usable?" for whatever reads `HLS_TINDEX_FN_DICT` (so it works for any year in `mosaiclib`, not only `YEAR_LIST`). Reads overviews only, so it is quick for tens of tiles. `problems` flags: not in the tindex, duplicate tindex rows, file cannot be opened, not 3000 × 3000, raster does not cover the tile polygon (< 95%), mostly nodata (< 50% valid). Comparing a problem year against a year that worked (e.g. 2024 vs 2025) shows whether the comp itself is the difference.

In [ ]:
CHECK_TILES = sorted(LARGE_DATELINE_TILES + MINI_DATELINE_TILES)   # default: the 18 tiles touching ±180°
CHECK_YEARS = ['2024', '2025']                                    # keys of mosaiclib.HLS_TINDEX_FN_DICT
DO_CHECK = True
if DO_CHECK:
    comp_check = check_composites(CHECK_TILES, CHECK_YEARS, grid)
    display(comp_check.pivot_table(index='tile_num', columns='year', values=['valid_frac', 'tile_coverage', 'n_obs_median'], aggfunc='first'))
    display(comp_check[comp_check.problems != ''][['tile_num', 'year', 'problems', 'width', 'height', 'crs', 'error', 'creation_time', 's3_path']])

In [ ]:
# Optional: false-color quick look (SWIR / NIR / Red) of one tile across years
SHOW_TILE = CHECK_TILES[0]
show_comps(SHOW_TILE, CHECK_YEARS)

---
## Appendix: reference calls (not part of the run)

### A. Register the algorithm (only after changing the yml or the code)
1. Commit, then tag the commit with the algorithm version and push the tag (DPS builds from that git ref):
   `git tag -f HLS_stack_2023_v1` → `git push` → `git push origin -f HLS_stack_2023_v1`. Check `git log` shows the tag at `origin`.
2. The yml (`dps/registered/do_HLS_stack_3-1-2.yml` or `run_build_ms_composite.yml`) must name that version and the GitHub repository; pin package versions in its build command so the environment is stable.
3. Register with the cell below.

In [ ]:
ALGO_YML = {'do_HLS_stack_3-1-2': 'do_HLS_stack_3-1-2.yml', 'run_build_ms_composite': 'run_build_ms_composite.yml'}
SHOW_REFERENCE = False
if SHOW_REFERENCE:
    print([a for a in maap.listAlgorithms().json()['algorithms'] if a['type'] == ALGO])
    print(json.dumps(get_in_param_dict(TILES[0], YEAR_LIST[0]), indent=1, default=str))
    # print(maap.register_algorithm_from_yaml_file(f'{ICESAT2_BOREAL_REPO_PATH}/dps/registered/{ALGO_YML[ALGO]}').text)

### B. Local test: does the compositor run for one tile here?
Same arguments the DPS job gets. For `do_HLS_stack_3-1-2`, `LOCAL_SEARCH_ONLY = True` stops after the CMR search and writes the search response JSON (no composite); `False` builds the composite here (slow, memory heavy). `run_build_ms_composite` has no search-only mode.

In [ ]:
DO_LOCAL_TEST = False
LOCAL_TEST_TILE = CHECK_TILES[0]
LOCAL_SEARCH_ONLY = True
LOCAL_OUTDIR = '/projects/my-private-bucket/tmp'
if DO_LOCAL_TEST:
    script = {'do_HLS_stack_3-1-2': '3.1.2_dps.py', 'run_build_ms_composite': 'build_ms_composite.py'}[ALGO]
    p = get_in_param_dict(LOCAL_TEST_TILE, YEAR_LIST[0])
    cmd = ['python', f'{ICESAT2_BOREAL_REPO_PATH}/lib/{script}', '-i', GRID['url'], '-lyr', GRID['layer'], '-a', p['sat_api'],
           '--tile_buffer_m', str(p['tile_buffer_m']), '--in_tile_num', str(LOCAL_TEST_TILE), '-o', LOCAL_OUTDIR,
           '-sy', p['start_year'], '-ey', p['end_year'], '-smd', p['start_month_day'], '-emd', p['end_month_day'],
           '-mc', str(p['max_cloud']), '--composite_type', p['composite_type'], '--hls_product', p['hls_product'],
           '--shape', str(p['shape']), '--min_n_filt_results', str(p['min_n_filt_results'])]
    if ALGO == 'run_build_ms_composite':   # this script has no --search_only
        cmd += ['--thresh_min_ndvi', str(p['thresh_min_ndvi']), '--stat', p['stat'], '--stat_pct', str(p['stat_pct']),
                '--target_spectral_index', p['target_spectral_index']]
    elif LOCAL_SEARCH_ONLY:
        cmd.append('--search_only')
    print(' '.join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout[-4000:], res.stderr[-4000:])

### C. Maps from the composite parameters table
Needs `HLS_input_params.csv` (section 7 with `WRITE_PARAMS_TABLE = True`). `run_type` = the season/cloud parameters that made each tile's comp; `num_scenes` = scenes that went into it.

In [ ]:
SHOW_PARAMS_YEAR = None   # e.g. '2025'
if SHOW_PARAMS_YEAR:
    params = pd.read_csv(tindex_paths(SHOW_PARAMS_YEAR)['params'])
    gp = grid.merge(params[['tile_num', 'run_type', 'num_scenes']], on='tile_num', how='left')
    display(gp.groupby('run_type', dropna=False).size().rename('n_tiles'))
    print(f'{gp.num_scenes.isna().sum()} grid tiles without a params row')
    ExtractUtils.plot_gdf_on_world(gp)
    ExtractUtils.plot_gdf_on_world(gp, MAP_COL='num_scenes')

### D. Delete the DPS dirs of duplicate (superseded) comps
After a rebuild, `HLS_tindex_master_duplicates.csv` lists the older comps of tiles that have a newer one. Deleting their job dirs frees space and keeps a later rebuild from ever picking them. **This cannot be undone.** Do it only after checking the newer comps (section 8). Thousands of dirs take hours.

In [ ]:
DELETE_DUPS_YEAR = None      # e.g. '2025'
CONFIRM_DELETE = False       # must be True to delete
if DELETE_DUPS_YEAR:
    dups = pd.read_csv(tindex_paths(DELETE_DUPS_YEAR)['dups'])
    dirs = sorted({os.path.dirname(p) for p in dups.s3_path})
    print(f'{len(dirs)} job dirs of superseded comps in {MS_COMP_NAMES[DELETE_DUPS_YEAR]}, e.g. {dirs[0] if dirs else None}')
    if CONFIRM_DELETE:
        for i, d in enumerate(dirs, 1):
            subprocess.run(['aws', 's3', 'rm', '--quiet', '--recursive', d])
            if i % 100 == 0:
                print(f'  {i}/{len(dirs)} deleted')
        print('done')